# SRGAN ×3 — 판별자 붕괴 전후 비교

흐린 위성사진(10 m) → 3배 선명하게(3.33 m).
학습 도중 판별자가 무너지는데, **무너지기 전과 후 중 어느 쪽이 더 좋은지** 직접 비교한다.

## 1. 데이터

In [ ]:
import urllib.request
LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'srgan_models.py', 'srgan_losses.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)

from sr_utils import *

val_lr, val_hr = pair('validation', REP['validation'])
test_lr = load_test()
show([('validation (Paris)', val_lr, val_hr), ('test (Incheon)', test_lr, None)])

## 2. 훈련

**코드가 도는지 확인하는 용도다.** 적은 데이터로 몇 번만 돌린다.
아래 3~5번은 전체 데이터로 100 epoch 학습해둔 가중치를 쓴다.

생성자와 판별자를 번갈아 갱신하는 것이 GAN 학습의 전부다.

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from srgan_models import Generator, Discriminator
from srgan_losses import GeneratorLoss

N_TRAIN, EPOCHS, BATCH = 16, 3, 4
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

lo, hi = zip(*[pair('training', s) for s in list_split('training')[:N_TRAIN]])
to_t = lambda a: torch.from_numpy(np.stack(a).transpose(0, 3, 1, 2)).float() / 255
loader = DataLoader(TensorDataset(to_t(lo), to_t(hi)), batch_size=BATCH, shuffle=True)

netG, netD = Generator(3).to(dev).train(), Discriminator().to(dev).train()
optG, optD = torch.optim.Adam(netG.parameters()), torch.optim.Adam(netD.parameters())
crit = GeneratorLoss().to(dev)

for ep in range(1, EPOCHS + 1):
    gl = dl = dx = dgz = 0.0
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        fake = netG(x)                                     # 1) 생성자
        g_loss = crit(netD(fake).mean(), fake, y)
        optG.zero_grad(); g_loss.backward(); optG.step()

        real_out, fake_out = netD(y).mean(), netD(fake.detach()).mean()
        d_loss = 1 - real_out + fake_out                   # 2) 판별자
        optD.zero_grad(); d_loss.backward(); optD.step()

        gl += g_loss.item(); dl += d_loss.item()
        dx += real_out.item(); dgz += fake_out.item()
    k = len(loader)
    print(f'epoch {ep}/{EPOCHS}  Loss_G {gl/k:.4f}  Loss_D {dl/k:.4f}  '
          f'D(x) {dx/k:.3f}  D(G(z)) {dgz/k:.3f}')

## 3. 결과 비교 — 붕괴 전 vs 붕괴 후

전체 데이터로 100 epoch 돌린 기록이다. **epoch 37 에서 판별자가 무너졌다.**

In [ ]:
import pandas as pd

MODEL = f'{BASE}/models/02_srgan_x3'
e = pd.read_csv(fetch(f'{MODEL}/statistics/train_results.csv', 'log.csv'), index_col=0)
COLLAPSE = 37

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(e.index, e.Score_D, color='#2f6f9f', lw=1.6, label='D(x)  real')
ax[0].plot(e.index, e.Score_G, color='#c96a5b', lw=1.6, label='D(G(z))  fake')
ax[0].axvline(COLLAPSE, color='#333', ls='--', lw=1)
ax[0].set_title(f'Discriminator output — collapses to 1 at epoch {COLLAPSE}')
ax[0].set_ylim(-.05, 1.08); ax[0].legend(fontsize=8)

ax[1].plot(e.index, e.Loss_D, color='#4f9d69', lw=1.6)
ax[1].axhline(1.0, ls='--', c='#888', lw=1)
ax[1].set_title('Discriminator loss — flat, gives no warning')
ax[1].set_ylim(.85, 1.06)

ax[2].plot(e.index, e.w_adversarial, color='#c96a5b', lw=1.6)
ax[2].axvline(COLLAPSE, color='#333', ls='--', lw=1)
ax[2].set_title('Weighted adversarial term — pinned at 0')
for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f'붕괴 후 D(x) = {e.iloc[-1].Score_D:.4f}, D(G(z)) = {e.iloc[-1].Score_G:.4f}'
      f'  ->  모든 입력을 "진짜" 로 판정한다')
print(f'Loss_D = {e.iloc[-1].Loss_D:.4f}  ->  1 - D(x) + D(G(z)) 라 붕괴해도 1.0 이다')

In [ ]:
from srgan_models import load_srgan

CK = {'before (ep29)': 'srgan_g_x3_ep29_before.pth',
      'after  (ep70)': 'srgan_g_x3_ep70_after.pth'}
nets = {k: load_srgan(fetch(f'{MODEL}/checkpoints/{v}', v)) for k, v in CK.items()}

def make(net):
    @torch.no_grad()
    def f(lr):
        t = torch.from_numpy(lr.transpose(2, 0, 1)).float()[None].to(next(net.parameters()).device) / 255
        return (net(t).clamp(0, 1)[0].cpu().numpy().transpose(1, 2, 0) * 255).round().astype('uint8')
    return f

up = {k: make(v) for k, v in nets.items()}
zoom([('Bicubic', bicubic(val_lr))] + [(k, up[k](val_lr)) for k in CK] + [('Target HR', val_hr)],
     title='validation (Paris), x3')

## 4. 평가

In [ ]:
for k in CK:
    print(f'--- {k} ---')
    compare(up[k], label=f'SRGAN {k}', plot=False)
    print()

In [ ]:
rows = compare(up['after  (ep70)'], label='SRGAN after')

붕괴 **후** 가중치가 더 좋다.

적대적 항이 0 으로 고정되어 기울기가 사라진 뒤로는 사실상 MSE + VGG 손실만으로
학습되는데, 이 데이터에서는 그쪽이 PSNR·SSIM 에 더 유리했다.
**GAN 이 항상 이득은 아니다.**

## 5. 최종 테스트 — 인천

정답이 없는 실제 Sentinel-2 촬영본이다. 점수는 못 내고 눈으로 확인한다.
앞에서 더 좋았던 붕괴 후 가중치만 쓴다.

In [ ]:
BEST = 'after  (ep70)'
t_bic = bicubic(test_lr)
t_sr = up[BEST](test_lr)

# 확대 위치는 장면(bicubic) 기준으로 고른다. 모델 출력을 기준으로 삼으면
# 모델이 바뀔 때마다 보는 곳이 달라져 비교가 안 된다.
zoom([('Bicubic', t_bic), ('SRGAN (ep70)', t_sr)], ref=t_bic,
     title='test (Incheon), x3 - no target')

def sharpness(a):
    return float(cv2.Laplacian(cv2.cvtColor(a, cv2.COLOR_RGB2GRAY), cv2.CV_64F).std())

print(f'{"":18s}{"sharpness":>11s}{"mean RGB":>22s}')
print(f'{"Bicubic":18s}{sharpness(t_bic):11.2f}{str(t_bic.reshape(-1,3).mean(0).round(1)):>22s}')
print(f'{"SRGAN (ep70)":18s}{sharpness(t_sr):11.2f}{str(t_sr.reshape(-1,3).mean(0).round(1)):>22s}')

imageio.imwrite('incheon_srgan.png', t_sr)
print('\nincheon_srgan.png 저장')